# 02 - Methode 1 : separabilite sans visualisation

IFT599-1 / IFT799-51 / IFT799-52 - TP1

Membres : Nom Prenom - CIP

Objectif : mesurer l'Overlap des 10 paires de classes, avec la distance
Euclidienne puis celle de Mahalanobis, sur toutes les variables de l'ensemble
de depart. Les resultats sont presentes sous forme de tableaux.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from itertools import combinations

import pandas as pd

from src.config import CLASSES
from src.loading import charger
from src.distances import euclidienne, mahalanobis, sigma_inverse
from src.separability import dist_intra, dist_inter, overlap

In [ ]:
def tableau(X, y, dist):
    lignes = []
    for a, b in combinations(CLASSES, 2):
        Ca, Cb = X[y == a], X[y == b]
        lignes.append({
            "paire": f"{a}-{b}",
            "intra_1": dist_intra(Ca, dist),
            "intra_2": dist_intra(Cb, dist),
            "inter": dist_inter(Ca, Cb, dist),
            "overlap": overlap(Ca, Cb, dist),
        })
    t = pd.DataFrame(lignes).round(3)
    t["bien_separees"] = t["overlap"] < 1
    return t

## Distance Euclidienne

In [ ]:
X, y = charger()

t_eucl = tableau(X, y, euclidienne)
t_eucl.to_csv("../results/method1_euclidienne.csv", index=False)
t_eucl

## Distance de Mahalanobis

La covariance est poolee : une seule matrice Sigma estimee sur les 801 patients,
commune aux cinq classes. Les deux metriques portent donc sur les memes
750 variables.

In [ ]:
maha = mahalanobis(sigma_inverse(X))

t_maha = tableau(X, y, maha)
t_maha.to_csv("../results/method1_mahalanobis.csv", index=False)
t_maha

## Comparaison Euclidienne / Mahalanobis

Les deux metriques sont calculees sur les memes 750 variables : seule la mesure
de distance change.

In [ ]:
comp = pd.DataFrame({
    "paire": t_eucl["paire"],
    "euclidienne": t_eucl["overlap"],
    "mahalanobis": t_maha["overlap"],
})
comp.to_csv("../results/method1_comparaison.csv", index=False)
comp

### Commentaires

Une seule paire passe sous le seuil : COAD-PRAD (0.994) en euclidien. C'est
aussi la seule dont la distance inter-classe depasse les deux distances
intra-classes.

Les autres paires restent au-dessus de 1, sans que les classes se confondent
pour autant. La distance intra-classe etant un **maximum**, un seul patient
atypique suffit a gonfler le rayon de toute sa classe. L'enonce precise d'ailleurs
que Overlap < 1 est suffisant mais non necessaire.

Mahalanobis donne des valeurs plus elevees et surtout beaucoup plus resserrees :
de 1.036 a 1.099, soit une etendue de 0.06, contre 0.50 en euclidien. Les
distances intra-classes deviennent presque identiques d'une classe a l'autre
(27.8 a 28.2 contre 42.2 a 53.7).

En ramenant toutes les directions a une variance comparable, Mahalanobis efface
les differences d'echelle entre classes, y compris celles qui portaient
l'information discriminante.